# Fundación YPF - Data Science

## Entrega Final Unificada
**Alumnas:** Florencia Zoni, Virginia Contreras  

### Contenido del Cuaderno:
1. **Módulo 1:** Predicción de tasas de promoción en el nivel secundario mediante **Regresión Lineal Múltiple**.
2. **Módulo 2:** **Clustering** de jurisdicciones argentinas según sus trayectorias escolares (2012-2023).

---

# Módulo 1: Regresión Lineal Múltiple

## 1.1 Objetivo y Estrategia Metodológica
Predecir la tasa total de promoción de secundaria del año siguiente ($t+1$) a partir de las tasas registradas en el año actual ($t$).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
# Carga y limpieza de datos para regresión
ruta_archivo = 'Tasa_de_Promocion_Efectiva_2012-2023_unificada.xlsx'
df = pd.read_excel(ruta_archivo)
df_limpio_cols = df.dropna(axis=1, how='any').copy()
df_limpio_cols = df_limpio_cols.sort_values(by=['División político-territorial', 'Año']).reset_index(drop=True)

# Crear la variable objetivo t+1
df_limpio_cols['Secundaria_Total_Siguiente'] = df_limpio_cols.groupby('División político-territorial')['Secundaria - Total'].shift(-1)
df_modelo = df_limpio_cols.dropna(subset=['Secundaria_Total_Siguiente']).copy()

# Definir características predictoras
caracteristicas = [
    'Secundaria - Total',
    'Secundaria - 8° Año',
    'Secundaria - 9° Año',
    'Secundaria - 10° Año',
    'Secundaria - 11° Año',
    'Secundaria - 12° Año'
]

# División temporal (Train <= 2018, Test > 2018)
df_entrenamiento = df_modelo[df_modelo['Año'] <= 2018]
df_prueba = df_modelo[df_modelo['Año'] > 2018]

X_train = df_entrenamiento[caracteristicas]
y_train = df_entrenamiento['Secundaria_Total_Siguiente']
X_test = df_prueba[caracteristicas]
y_test = df_prueba['Secundaria_Total_Siguiente']

# Ajuste del modelo
regresion_multiple = LinearRegression()
regresion_multiple.fit(X_train, y_train)
y_pred_multi = regresion_multiple.predict(X_test)

# Métricas de evaluación
rmse = np.sqrt(mean_squared_error(y_test, y_pred_multi))
mae = mean_absolute_error(y_test, y_pred_multi)
r2 = r2_score(y_test, y_pred_multi)

print('--- RESULTADOS REGRESIÓN LINEAL MÚLTIPLE ---')
print(f'MAE: {mae:.2f} p.p.')
print(f'RMSE: {rmse:.2f} p.p.')
print(f'R2 Score: {r2:.2f}')

--- 
# Módulo 2: Clustering de Jurisdicciones Argentinas (2012-2023)

## 2.1 Preparación y Normalización de Datos

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOM_STATE = 42
df_clust = pd.read_excel(ruta_archivo)
df_clust.columns = ["anio", "jur", "estructura",
                    "Prim_Total", "Prim_1", "Prim_2", "Prim_3", "Prim_4", "Prim_5", "Prim_6", "Prim_7",
                    "Sec_Total", "Sec_7", "Sec_8", "Sec_9", "Sec_10", "Sec_11", "Sec_12"]

# Excluir regiones agregadas
agregados = ["Total País", "Conurbano", "Resto de Bs As"]
d_provincias = df_clust[~df_clust.jur.isin(agregados)]

grados = ["Prim_1", "Prim_2", "Prim_3", "Prim_4", "Prim_5", "Prim_6", "Sec_8", "Sec_9", "Sec_10", "Sec_11", "Sec_12"]

perfil = d_provincias.groupby("jur")[grados].mean()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(perfil)

# PCA y K-Means con k=3
pca = PCA(n_components=4, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

kmeans_final = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
perfil["cluster"] = kmeans_final.fit_predict(X_pca)

print('--- CLUSTERING FINALIZADO ---')
print(perfil["cluster"].value_counts())